# Лабораторна 2. Від сирих даних до нейромережі

**Студент:** Зайцев Денис Сергійович  
**Мій датасет:** Bike Sharing Demand (таргет — `cnt`, кількість орендованих велосипедів за годину).

Ноутбук запускається зверху донизу (Run All) в будь-якому середовищі (Colab / локально), на CPU.

In [ ]:
%pip install -q datasets tensorboard gradio scikit-learn

In [ ]:
import os, random, shutil, hashlib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.tensorboard import SummaryWriter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from IPython.display import Markdown, display

SEED = 42

def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
device = torch.device("cpu")   # датасет і моделі малі — CPU достатньо

MODELS_DIR = "models"
RUNS_DIR = "runs"
RESULTS_PATH = "results.csv"
shutil.rmtree(RUNS_DIR, ignore_errors=True)   # щоб повторний запуск не дублював логи
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RUNS_DIR, exist_ok=True)
print("torch", torch.__version__, "| device:", device)

---
## 0. Відповіді на запитання до демо-ноутбука

### 0.1 Мої відповіді

**1. Чому для заповнення пропусків обрано kNN?**  
Простіші методи (середнє/медіана) підставляють одне й те саме число всім пропускам: це ігнорує інші ознаки, «стискає» дисперсію і руйнує кореляції між ознаками. kNN-імпутація шукає рядки, найбільш схожі за *іншими* ознаками, і бере значення з них. Тому відновлене значення узгоджене з рештою рядка, зберігається локальна структура даних, не потрібно припускати лінійну залежність, і метод працює, коли пропущено кілька ознак одразу. Умови коректності: ознаки треба масштабувати (інакше відстань визначатиме ознака з найбільшою шкалою), а імпутер навчати **лише на train** (інакше data leakage). Недолік — повільніший за середнє і чутливий до вибору `k`.

**2. Чому під час тренування train і val порівнюються лише за MSE, а після — за кількома метриками?**  
MSE — це саме та функція, яку оптимізує градієнтний спуск (loss): вона диференційована, і її значення на train та val безпосередньо порівнянні, тому за розходженням кривих видно перенавчання. Рахувати всі метрики щоепохи зайве: RMSE — це корінь із MSE, R² — перемасштабований MSE (`1 − MSE/Var(y)`), тож нової інформації про збіжність вони не дають. MAE ж — не гладка в нулі, її градієнти інші. Повний набір метрик у фізичних одиницях потрібен вже для **фінальної** оцінки моделі на test. Експеримент із вибором функції втрат (MSE / MAE / Huber) на моїх даних — у розділі 2.2 («Експеримент: вибір loss»).

**3. Які висновки після клітинки 2.6 (інтерпретація ваг)?**  
Після стандартизації ознак ваги лінійної моделі можна порівнювати між собою: знак показує напрям впливу, модуль — силу (на скільки σ зміниться таргет при зміні ознаки на 1σ). Але ваги коректно інтерпретувати лише за відсутності сильної мультиколінеарності: корельовані ознаки «ділять» вплив між собою, і окремі ваги нестабільні. Інтерпретацію ваг для **моїх** даних я наводжу у розділі 2.2.

**4. Для чого потрібен baseline і як він вираховується? Що baseline для нейромережі?**  
Baseline — найпростіша модель, з якою порівнюють усі інші: без нього незрозуміло, чи хороше число метрики взагалі щось означає. Для регресії найпростіший baseline — завжди передбачати середнє таргета на train; його MSE дорівнює дисперсії таргета, а R² = 0. Модель, яка не б’є такий baseline, нічого не навчилася. Множинна лінійна регресія — вже «розумніший» baseline. Для нейромережі baseline — це лінійна регресія (і, ще нижче, середнє): нейромережа виправдана, лише якщо дає суттєво кращі метрики за лінійну модель. Обидва baseline обчислені в розділі 2.2.

**5. Функції активації на графіках** — клітинка нижче (1000 випадкових X, Y рахується через кожну функцію).

**6. Звідки береться кількість параметрів у нейромережі?**  
Кожен повнозв'язний шар `Linear(in, out)` має `in·out` ваг і `out` зсувів, тобто `in·out + out` параметрів. Функції активації параметрів не мають. Загальна кількість — сума по всіх шарах. Нижче я перераховую це для моїх архітектур і звіряю з `sum(p.numel())`.

**7. Чи оптимально вибрано кількість епох?**  
Оптимальна кількість епох — там, де мінімальна val-втрата: якщо криві val ще падають, то моделі не вистачило епох, якщо val росте при падінні train — це перенавчання, і потрібен early stopping. Це перевіряється за кривими навчання, тому для моїх моделей я роблю це явно в розділі 2.4 (епоха мінімуму val для кожного прогону).

In [ ]:
# Відповідь на питання 5: графіки функцій активації
set_seed()
x_rand = torch.empty(1000).uniform_(-5, 5)          # 1000 випадкових X
activations = {
    "ReLU": nn.ReLU(), "LeakyReLU(0.1)": nn.LeakyReLU(0.1), "ELU": nn.ELU(), "GELU": nn.GELU(),
    "SiLU (Swish)": nn.SiLU(), "Softplus": nn.Softplus(), "Sigmoid": nn.Sigmoid(), "Tanh": nn.Tanh(),
}
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
with torch.no_grad():
    for ax, (name, fn) in zip(axes.ravel(), activations.items()):
        y_act = fn(x_rand)                           # Y через функцію активації
        ax.scatter(x_rand.numpy(), y_act.numpy(), s=4)
        ax.axhline(0, color="gray", lw=0.5); ax.axvline(0, color="gray", lw=0.5)
        ax.set_title(name); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 1. Датасет

### 1.1. Який датасет мій?

In [ ]:
DATASETS = [
    "King County House Sales", "Diamonds", "Concrete Compressive Strength", "Auto MPG", "Wine Quality",
    "Laptop Price", "Bike Sharing Demand", "Ames House Prices", "Medical Insurance Cost", "Abalone",
]
FULL_NAME = "Зайцев Денис Сергійович"

dataset_index = int(hashlib.sha256(FULL_NAME.encode("utf-8")).hexdigest(), 16) % len(DATASETS)
print(f"Ваш датасет (#{dataset_index + 1}):", DATASETS[dataset_index])
assert DATASETS[dataset_index] == "Bike Sharing Demand"

---
## 2. Підготовка даних

### 2.1.1 Завантаження і спліт

**Чому спліт робимо до будь-якого аналізу.** Усе, що ми «дізнаємось» з даних (середні для стандартизації, кореляції, пороги викидів, сусіди для kNN, рішення про ознаки), має спиратися лише на train. Якщо подивитись на весь датасет до спліту, інформація про val/test непомітно просочується в рішення про модель (data leakage), і оцінка на test стає занадто оптимістичною. Тому val і test «замикаємо» одразу, а fit будь-яких перетворень робимо тільки на train.

> Примітка: Bike Sharing — погодинний часовий ряд, тому випадковий спліт дає сусідні години в train і test (автокореляція), і оцінки трохи оптимістичніші, ніж при хронологічному спліті. Я використовую випадковий спліт 70/15/15, як у демо; для порівняння моделей між собою це коректно.

In [ ]:
from datasets import load_dataset

ds = load_dataset("t22000t/bike-sharing-tabular")
print(ds)
df_raw = pd.concat([ds[s].to_pandas() for s in ds.keys()], ignore_index=True)
df_raw.columns = [c.strip() for c in df_raw.columns]

TARGET = "cnt"
assert TARGET in df_raw.columns, f"Немає таргета {TARGET}: {list(df_raw.columns)}"
n_before = len(df_raw)
df_raw = df_raw.dropna(subset=[TARGET]).reset_index(drop=True)
print(f"Рядків: {n_before} -> {len(df_raw)} (після видалення рядків без таргета)")

train_df, tmp_df = train_test_split(df_raw, test_size=0.30, random_state=SEED)
val_df, test_df = train_test_split(tmp_df, test_size=0.50, random_state=SEED)
print(f"train/val/test: {len(train_df)}/{len(val_df)}/{len(test_df)}")

### 2.1.2 Загальний огляд (лише train)

In [ ]:
display(train_df.head())
train_df.info()
display(train_df.describe().T)

### 2.1.3 Пропуски

In [ ]:
missing = train_df.isna().sum()
print(missing[missing > 0] if missing.sum() else "Пропусків у train немає.")

if missing.sum() > 0:
    # kNN-імпутація: fit тільки на train
    from sklearn.impute import KNNImputer
    num_cols = train_df.select_dtypes("number").columns
    imputer = KNNImputer(n_neighbors=5).fit(train_df[num_cols])
    for d in (train_df, val_df, test_df):
        d[num_cols] = imputer.transform(d[num_cols])
    print("Пропуски заповнено kNN (fit на train).")
else:
    print("Пропусків немає — імпутація не потрібна.")
print("Пропуски у val/test:", int(val_df.isna().sum().sum()), "/", int(test_df.isna().sum().sum()))

Пропусків немає — тому імпутацію не виконуємо (штучно їх вигадувати не потрібно).

### 2.1.4 EDA: гістограми і кореляції

In [ ]:
num_cols = train_df.select_dtypes("number").columns.tolist()
train_df[num_cols].hist(bins=30, figsize=(16, 11))
plt.tight_layout(); plt.show()

# Перевірка на витік таргета: cnt = casual + registered?
if {"casual", "registered"} <= set(train_df.columns):
    share = (train_df["casual"] + train_df["registered"] == train_df[TARGET]).mean()
    print(f"Частка рядків, де casual + registered == cnt: {share:.3f}")

In [ ]:
corr = train_df[num_cols].corr()
fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(num_cols))); ax.set_xticklabels(num_cols, rotation=90)
ax.set_yticks(range(len(num_cols))); ax.set_yticklabels(num_cols)
plt.colorbar(im); plt.title("Кореляційна матриця (train)"); plt.tight_layout(); plt.show()

print("Кореляція з таргетом:")
print(corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).round(3))

pairs = [(a, b, corr.loc[a, b]) for i, a in enumerate(num_cols) for b in num_cols[i + 1:]
         if abs(corr.loc[a, b]) > 0.9 and TARGET not in (a, b)]
print("\nПари ознак із |corr| > 0.9:")
for a, b, c in pairs: print(f"  {a} ~ {b}: {c:.3f}")

**Що видно в даних (див. виводи вище):**
- `casual` і `registered` у сумі дають `cnt` — це **витік таргета** (ці ознаки недоступні, коли ми хочемо прогнозувати `cnt`), їх треба прибрати. `instant` та `dteday` — службові ідентифікатори/дата, які не є ознаками.
- `temp` та `atemp` майже ідеально корельовані (мультиколінеарність) — одну з них залишаємо. `season` дублюється `mnth`.
- Таргет `cnt` сильно скошений праворуч (багато годин із малим попитом і довгий хвіст піків); `hr` — циклічна ознака (23 і 0 години сусідні), з нелінійною залежністю (піки зранку й увечері).

### 2.1.5 Викиди

In [ ]:
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())

cont = [c for c in ["temp", "atemp", "hum", "windspeed", TARGET] if c in train_df.columns]
print("Викиди за IQR (train):", {c: iqr_outliers(train_df[c]) for c in cont})
for c in ["hum", "windspeed"]:
    if c in train_df.columns:
        print(f"{c} == 0: {(train_df[c] == 0).sum()} рядків")
print("Skew cnt:", round(train_df[TARGET].skew(), 2))

train_df[cont].plot(kind="box", subplots=True, layout=(1, len(cont)), figsize=(16, 4))
plt.tight_layout(); plt.show()

**Рішення щодо викидів.** Високі значення `cnt` — це справжні піки попиту (години пік у теплі робочі дні), а не помилки вимірювання, тому їх **не видаляємо**: інакше модель не вчилася б передбачати саме найцікавіші (і найдорожчі помилками) випадки. Нулі в `hum`/`windspeed` (якщо є) — найімовірніше, закодовані пропуски датчика, але їх мало/вони не руйнують навчання, а стандартизація і NN до них стійкі, тому залишаємо. Таргет не логарифмуємо, щоб метрики MSE/RMSE/MAE залишалися в природних одиницях «велосипедів на годину».

### 2.1.6 Інженерія ознак
Що робимо (усі перетворення детерміновані і не залежать від значень інших рядків, тому витоку немає):
1. Прибираємо `casual`, `registered` (витік), `instant`, `dteday` (службові).
2. Прибираємо `atemp` (мультиколінеарність із `temp`) та `season` (дублює `mnth`).
3. Циклічні ознаки `hr`, `mnth`, `weekday` кодуємо парою `sin/cos`, щоб 23:00 і 00:00 були «поруч».
4. `weathersit` — категоріальна, кодуємо one-hot (фіксований набір категорій 1–4, щоб колонки збігалися у train/val/test).

In [ ]:
REQUIRED = ["yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit", "temp", "hum", "windspeed"]
missing_cols = [c for c in REQUIRED if c not in train_df.columns]
assert not missing_cols, f"У датасеті немає колонок {missing_cols}; наявні: {list(train_df.columns)}"

def make_features(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    for col, period in [("hr", 24), ("mnth", 12), ("weekday", 7)]:
        ang = 2 * np.pi * df[col].astype(float) / period
        out[col + "_sin"], out[col + "_cos"] = np.sin(ang), np.cos(ang)
    for col in ["yr", "holiday", "workingday", "temp", "hum", "windspeed"]:
        out[col] = df[col].astype(float)
    for k in (1, 2, 3, 4):
        out[f"weather_{k}"] = (df["weathersit"] == k).astype(float)
    return out

X_train_df, X_val_df, X_test_df = (make_features(d) for d in (train_df, val_df, test_df))
FEATURES = X_train_df.columns.tolist()
N_FEATURES = len(FEATURES)
print(N_FEATURES, "ознак:", FEATURES)

---
## 2.2. Baseline: множинна лінійна регресія (варіант #1)

### Стандартизація і тензори
`StandardScaler` навчаємо тільки на train. Таргет теж стандартизуємо за статистиками train (так градієнтний спуск стабільніший), а всі метрики рахуємо після зворотного перетворення — в природних одиницях.

In [ ]:
scaler = StandardScaler().fit(X_train_df)
Y_MEAN, Y_STD = float(train_df[TARGET].mean()), float(train_df[TARGET].std())

def to_x(df): return torch.tensor(scaler.transform(df), dtype=torch.float32)
def to_y(s):  return torch.tensor(((s.values - Y_MEAN) / Y_STD), dtype=torch.float32).unsqueeze(1)

Xtr, Xva, Xte = to_x(X_train_df), to_x(X_val_df), to_x(X_test_df)
ytr, yva, yte = to_y(train_df[TARGET]), to_y(val_df[TARGET]), to_y(test_df[TARGET])
yte_np = test_df[TARGET].values.astype(float)
yva_np = val_df[TARGET].values.astype(float)
print(Xtr.shape, Xva.shape, Xte.shape)

def regression_metrics(y_true, y_pred):
    err = y_true - y_pred
    mse = float(np.mean(err ** 2))
    return {"MSE": mse, "RMSE": mse ** 0.5, "MAE": float(np.mean(np.abs(err))),
            "R2": float(1 - np.sum(err ** 2) / np.sum((y_true - y_true.mean()) ** 2))}

@torch.no_grad()
def predict_original(model, X):
    model.eval()
    return model(X).squeeze(1).numpy() * Y_STD + Y_MEAN

# Naive baseline: завжди передбачати середнє train
baseline_mean = regression_metrics(yte_np, np.full_like(yte_np, train_df[TARGET].mean()))
print("Baseline «середнє train» на test:", {k: round(v, 3) for k, v in baseline_mean.items()})

### Універсальна функція тренування (з логуванням у TensorBoard)

In [ ]:
RESULTS, HISTORIES, FACTORIES, VAL_MSE, DESCR = [], {}, {}, {}, {}

def make_optimizer(kind, params, lr):
    return {"adam": torch.optim.Adam, "sgd": torch.optim.SGD}[kind](params, lr=lr)

def train_model(name, model, optimizer, epochs, batch_size=256, loss_fn=None, track=True):
    loss_fn = loss_fn or nn.MSELoss()
    writer = SummaryWriter(os.path.join(RUNS_DIR, name)) if track else None
    g = torch.Generator().manual_seed(SEED)
    hist = {"train_rmse": [], "val_rmse": []}
    n = len(Xtr)
    for epoch in range(epochs):
        model.train()
        perm = torch.randperm(n, generator=g)
        total = 0.0
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            optimizer.zero_grad()
            loss = loss_fn(model(Xtr[idx]), ytr[idx])
            loss.backward()
            optimizer.step()
            total += loss.item() * len(idx)
        model.eval()
        with torch.no_grad():
            train_mse = nn.functional.mse_loss(model(Xtr), ytr).item()
            val_mse = nn.functional.mse_loss(model(Xva), yva).item()
        hist["train_rmse"].append(train_mse ** 0.5 * Y_STD)
        hist["val_rmse"].append(val_mse ** 0.5 * Y_STD)
        if writer:
            writer.add_scalar("loss/train_mse_std", train_mse, epoch)
            writer.add_scalar("loss/val_mse_std", val_mse, epoch)
            writer.add_scalar("rmse/val_original", hist["val_rmse"][-1], epoch)
    if writer: writer.close()

    test_m = regression_metrics(yte_np, predict_original(model, Xte))
    val_m = regression_metrics(yva_np, predict_original(model, Xva))
    if track:
        torch.save(model.state_dict(), os.path.join(MODELS_DIR, f"{name}.pt"))
        HISTORIES[name] = hist
        VAL_MSE[name] = val_m["MSE"]
        RESULTS.append({"model": name, **test_m})
    return test_m, hist

def run_variant(name, factory, descr, optimizer="adam", lr=1e-3, epochs=60, batch_size=256):
    set_seed()
    model = factory()
    FACTORIES[name], DESCR[name] = factory, descr
    test_m, hist = train_model(name, model, make_optimizer(optimizer, model.parameters(), lr), epochs, batch_size)
    n_par = sum(p.numel() for p in model.parameters())
    print(f"{name:22s} params={n_par:6d}  test RMSE={test_m['RMSE']:.2f}  MAE={test_m['MAE']:.2f}  R2={test_m['R2']:.4f}")
    return model

### Варіант #1: лінійна регресія (`nn.Linear`, SGD)

In [ ]:
linear_model = run_variant("linear", lambda: nn.Linear(N_FEATURES, 1),
                           "Linear(n_features, 1), SGD lr=0.01, 40 епох, batch 256",
                           optimizer="sgd", lr=0.01, epochs=40)

# Перевірка: аналітичний розв'язок (МНК) має дати майже ті самі метрики
from sklearn.linear_model import LinearRegression
ols = LinearRegression().fit(scaler.transform(X_train_df), train_df[TARGET].values)
ols_m = regression_metrics(yte_np, ols.predict(scaler.transform(X_test_df)))
print("МНК (sklearn) на test:", {k: round(v, 3) for k, v in ols_m.items()})

In [ ]:
h = HISTORIES["linear"]
plt.plot(h["train_rmse"], label="train"); plt.plot(h["val_rmse"], label="val")
plt.xlabel("epoch"); plt.ylabel("RMSE (велосипедів/год)"); plt.title("Крива навчання: linear")
plt.legend(); plt.grid(alpha=0.3); plt.show()

### Інтерпретація ваг лінійної моделі (2.6)

In [ ]:
w = pd.Series(linear_model.weight.detach().numpy().ravel(), index=FEATURES)
print("Зсув b =", round(linear_model.bias.item(), 4))
w.sort_values(key=abs).plot(kind="barh", figsize=(7, 6), title="Ваги (стандартизовані ознаки → стандартизований cnt)")
plt.grid(alpha=0.3); plt.show()
print(w.sort_values(key=abs, ascending=False).round(3))

### Експеримент: вибір функції втрат (до питання 2)

In [ ]:
loss_rows = []
for lname, lf in [("MSELoss", nn.MSELoss()), ("L1Loss (MAE)", nn.L1Loss()), ("HuberLoss", nn.HuberLoss(delta=1.0))]:
    set_seed()
    m = nn.Linear(N_FEATURES, 1)
    tm, _ = train_model(f"tmp_{lname}", m, torch.optim.SGD(m.parameters(), lr=0.01), 40, loss_fn=lf, track=False)
    loss_rows.append({"loss": lname, **{k: round(v, 3) for k, v in tm.items()}})
loss_df = pd.DataFrame(loss_rows)
display(loss_df)

best_rmse = loss_df.loc[loss_df["RMSE"].idxmin(), "loss"]
best_mae = loss_df.loc[loss_df["MAE"].idxmin(), "loss"]
display(Markdown(
    f"**Висновок з експерименту.** Найменший RMSE на test дає `{best_rmse}`, найменший MAE — `{best_mae}`. "
    "Функція втрат визначає, яку метрику модель оптимізує напряму: MSE штрафує великі помилки квадратично "
    "(чутлива до піків попиту), MAE/Huber стійкіші до викидів. Тому метрика, за якою ми оцінюємо, має "
    "узгоджуватись із loss, а порівнювати train і val коректно саме за loss, який оптимізується."
))

---
## 2.3. Нейромережа: 3 різні архітектури
Усі: Adam, lr = 1e-3, batch 256, 60 епох, MSE. Відмінності:
- `nn_v1` — 2 приховані шари (64-32), ReLU;
- `nn_v2` — глибша і ширша: 4 приховані шари (256-128-64-32), ReLU;
- `nn_v3` — як `nn_v1`, але з активацією Tanh (ізольований вплив функції активації).

In [ ]:
def mlp(hidden, act):
    def factory():
        layers, d = [], N_FEATURES
        for h_ in hidden:
            layers += [nn.Linear(d, h_), act()]
            d = h_
        layers.append(nn.Linear(d, 1))
        return nn.Sequential(*layers)
    return factory

ARCHS = {
    "nn_v1": (mlp([64, 32], nn.ReLU), "MLP 64-32, ReLU"),
    "nn_v2": (mlp([256, 128, 64, 32], nn.ReLU), "MLP 256-128-64-32, ReLU"),
    "nn_v3": (mlp([64, 32], nn.Tanh), "MLP 64-32, Tanh"),
}
for name, (factory, descr) in ARCHS.items():
    run_variant(name, factory, descr + ", Adam lr=1e-3, 60 епох, batch 256")

# Вибір найкращої архітектури — за VAL, а не за test
BEST_ARCH = min(ARCHS, key=lambda n: VAL_MSE[n])
print("\nНайкраща архітектура за val MSE:", BEST_ARCH, "-", ARCHS[BEST_ARCH][1])

In [ ]:
# Питання 6: звідки беруться параметри
for name in ["linear", *ARCHS]:
    m = FACTORIES[name]()
    parts, total = [], 0
    for layer in (m.modules() if not isinstance(m, nn.Linear) else [m]):
        if isinstance(layer, nn.Linear):
            p = layer.in_features * layer.out_features + layer.out_features
            parts.append(f"{layer.in_features}*{layer.out_features}+{layer.out_features}={p}")
            total += p
    assert total == sum(p.numel() for p in m.parameters())
    print(f"{name:8s}: " + " + ".join(parts) + f" = {total}")

---
## 2.4. Підбір гіперпараметрів (по одному за раз)
Беру найкращу за val архітектуру (`BEST_ARCH`) і змінюю **один** параметр відносно базової конфігурації (Adam, lr = 1e-3, batch 256, 60 епох):
learning rate (0.01 і 0.0001), оптимізатор (SGD із тим самим lr), розмір batch (64), кількість епох (150).

In [ ]:
factory, descr = ARCHS[BEST_ARCH]
base = dict(optimizer="adam", lr=1e-3, epochs=60, batch_size=256)
HP_VARIANTS = {
    f"{BEST_ARCH}_lr_0.01":   dict(base, lr=1e-2),
    f"{BEST_ARCH}_lr_0.0001": dict(base, lr=1e-4),
    f"{BEST_ARCH}_sgd":       dict(base, optimizer="sgd"),
    f"{BEST_ARCH}_bs_64":     dict(base, batch_size=64),
    f"{BEST_ARCH}_ep_150":    dict(base, epochs=150),
}
for name, cfg in HP_VARIANTS.items():
    changed = {k: v for k, v in cfg.items() if v != base[k]}
    run_variant(name, factory, f"{descr}; змінено: {changed}", **cfg)

In [ ]:
# Питання 7: чи оптимальна кількість епох?
rows = []
for name, h in HISTORIES.items():
    if name == "linear": continue
    v = np.array(h["val_rmse"])
    rows.append({"model": name, "epochs": len(v), "best_val_epoch": int(v.argmin()) + 1,
                 "min_val_RMSE": round(v.min(), 2), "final_val_RMSE": round(v[-1], 2)})
ep_df = pd.DataFrame(rows); display(ep_df)

late = ep_df[ep_df["best_val_epoch"] < ep_df["epochs"] * 0.8]
still = ep_df[ep_df["best_val_epoch"] >= ep_df["epochs"] * 0.95]
display(Markdown(
    f"**Висновок.** Прогони, де мінімум val досягнуто задовго до кінця (можливе перенавчання або зайві епохи): "
    f"{', '.join(late['model']) if len(late) else 'немає'}. "
    f"Прогони, де val ще падав майже до останньої епохи (епох могло бракувати): "
    f"{', '.join(still['model']) if len(still) else 'немає'}. "
    "Тому єдина «оптимальна» кількість епох для всіх моделей не існує — її слід обирати за кривою val "
    "(або використовувати early stopping)."
))

---
## 2.5. Логування і порівняння
Усі прогони записані в `runs/<назва_варіанта>/`. Щоб відкрити криві в TensorBoard (в Colab/Jupyter):
```
%load_ext tensorboard
%tensorboard --logdir runs
```
Нижче ті самі криві накладено одна на одну за допомогою matplotlib (з накопичених історій).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
for name, h in HISTORIES.items():
    axes[0].plot(h["train_rmse"], label=name)
    axes[1].plot(h["val_rmse"], label=name)
axes[0].set_title("Train RMSE"); axes[1].set_title("Val RMSE")
for ax in axes: ax.set_xlabel("epoch"); ax.set_ylabel("RMSE (велосипедів/год)"); ax.grid(alpha=0.3)
axes[1].legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
results_df = pd.DataFrame(RESULTS)
results_df.to_csv(RESULTS_PATH, index=False)
print("Збережено:", RESULTS_PATH)
display(results_df.round(4))
print("Baseline «середнє train»:", {k: round(v, 3) for k, v in baseline_mean.items()})

results_df.set_index("model")["RMSE"].plot(kind="bar", figsize=(11, 4), title="Test RMSE")
plt.axhline(baseline_mean["RMSE"], color="red", ls="--", label="baseline: середнє")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Прогноз vs факт: лінійна модель і найкраща за test
res = results_df.set_index("model")
winner = res["RMSE"].idxmin()
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
for ax, name in zip(axes, ["linear", winner]):
    m = FACTORIES[name](); m.load_state_dict(torch.load(os.path.join(MODELS_DIR, f"{name}.pt")))
    ax.scatter(yte_np, predict_original(m, Xte), s=4, alpha=0.3)
    lim = [0, yte_np.max()]; ax.plot(lim, lim, "r--")
    ax.set_title(f"{name}  (R²={res.loc[name, 'R2']:.3f})"); ax.set_xlabel("факт"); ax.set_ylabel("прогноз")
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### Письмові висновки (формуються за фактичними результатами цього прогону)

In [ ]:
arch_names = list(ARCHS)
hp_names = list(HP_VARIANTS)
lin = res.loc["linear"]
win = res.loc[winner]
imp = (1 - win["RMSE"] / lin["RMSE"]) * 100

best_arch_rmse = res.loc[BEST_ARCH, "RMSE"]
arch_gain = lin["RMSE"] - res.loc[arch_names, "RMSE"].min()
hp_best = res.loc[hp_names, "RMSE"].idxmin()
hp_gain = best_arch_rmse - res.loc[hp_best, "RMSE"]
hp_worst = res.loc[hp_names, "RMSE"].idxmax()
hp_loss = res.loc[hp_worst, "RMSE"] - best_arch_rmse

if winner == "linear":
    verdict = "Нейромережа **не виправдала себе**: лінійна регресія виявилась не гіршою."
elif imp >= 5:
    verdict = f"Нейромережа **виправдала себе**: RMSE менший приблизно на {imp:.0f}%."
else:
    verdict = f"Виграш нейромережі невеликий ({imp:.1f}% за RMSE), тож її складність виправдана лише частково."

biggest = ("зміна **архітектури** (лінійна → MLP)" if arch_gain >= max(hp_gain, hp_loss)
           else "підбір **гіперпараметрів**")
display(Markdown(f'''
- **Яка модель перемогла?** `{winner}`: RMSE = {win['RMSE']:.1f}, MAE = {win['MAE']:.1f}, R² = {win['R2']:.3f}
  проти лінійної регресії: RMSE = {lin['RMSE']:.1f}, MAE = {lin['MAE']:.1f}, R² = {lin['R2']:.3f}
  (baseline «середнє» має RMSE = {baseline_mean['RMSE']:.1f}, R² = {baseline_mean['R2']:.3f}).
  Різниця між кількома найкращими моделями може бути в межах шуму одного спліту — суттєвість варто оцінювати за порядком величини, а не за третім знаком.
- **Чи виправдала себе нейромережа?** {verdict}
  Фізичне пояснення: попит на велосипеди залежить від години доби нелінійно та в взаємодії з робочим/вихідним днем
  (піки о 8:00 і 17:00 у робочі дні, інший профіль у вихідні), чого лінійна модель на цих ознаках виразити не може.
- **Що дало найбільший ефект?** {biggest}: перехід від лінійної моделі до найкращої архітектури змінив RMSE на {arch_gain:.1f};
  найкращий гіперпараметричний варіант (`{hp_best}`) змінив RMSE відносно базової конфігурації `{BEST_ARCH}` на {hp_gain:+.1f},
  а найгірший (`{hp_worst}`) погіршив на {hp_loss:+.1f}.
'''))

---
## 3. Gradio-застосунок: порівняння всіх варіантів наживо
Застосунок завантажує всі збережені `.pt` з `models/`, приймає параметри години прокату і показує прогноз `cnt` від кожного варіанта (таблиця + стовпчикова діаграма).

In [ ]:
import gradio as gr

loaded = {}
for name in results_df["model"]:
    m = FACTORIES[name]()
    m.load_state_dict(torch.load(os.path.join(MODELS_DIR, f"{name}.pt")))
    m.eval(); loaded[name] = m
print("Завантажено моделей:", len(loaded))

def predict_all(hr, mnth, weekday, workingday, holiday, yr, weathersit, temp, hum, windspeed):
    raw = pd.DataFrame([{"hr": hr, "mnth": mnth, "weekday": weekday, "workingday": int(workingday),
                         "holiday": int(holiday), "yr": yr, "weathersit": int(weathersit),
                         "temp": temp, "hum": hum, "windspeed": windspeed}])
    x = torch.tensor(scaler.transform(make_features(raw)), dtype=torch.float32)
    rows = []
    for name, m in loaded.items():
        with torch.no_grad():
            pred = m(x).item() * Y_STD + Y_MEAN
        rows.append({"Модель": name, "Прогноз cnt": max(0, round(pred))})
    out = pd.DataFrame(rows)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(out["Модель"], out["Прогноз cnt"]); ax.set_ylabel("велосипедів/год")
    ax.tick_params(axis="x", rotation=60); ax.grid(alpha=0.3); plt.tight_layout()
    plt.close(fig)
    return out, fig

def rng(col): return float(train_df[col].min()), float(train_df[col].max())
yr_choices = sorted(train_df["yr"].unique().tolist())

demo = gr.Interface(
    fn=predict_all,
    inputs=[
        gr.Slider(0, 23, value=8, step=1, label="Година доби (hr)"),
        gr.Slider(1, 12, value=6, step=1, label="Місяць (mnth)"),
        gr.Slider(0, 6, value=2, step=1, label="День тижня (weekday)"),
        gr.Checkbox(value=True, label="Робочий день"),
        gr.Checkbox(value=False, label="Святковий день"),
        gr.Dropdown(yr_choices, value=yr_choices[-1], label="Рік (yr)"),
        gr.Dropdown([1, 2, 3, 4], value=1, label="Погода (weathersit: 1 ясно ... 4 зливи)"),
        gr.Slider(*rng("temp"), value=float(train_df["temp"].median()), label="Температура (temp)"),
        gr.Slider(*rng("hum"), value=float(train_df["hum"].median()), label="Вологість (hum)"),
        gr.Slider(*rng("windspeed"), value=float(train_df["windspeed"].median()), label="Швидкість вітру (windspeed)"),
    ],
    outputs=[gr.Dataframe(label="Прогнози всіх моделей"), gr.Plot(label="Порівняння")],
    title="Bike Sharing: порівняння моделей наживо",
    description="Лінійна регресія та нейромережі різних конфігурацій, навчені на Bike Sharing Demand.",
)
demo.launch()

### Перевірка структури файлів для здачі

In [ ]:
expected = [os.path.join(MODELS_DIR, f"{n}.pt") for n in results_df["model"]] + [RESULTS_PATH]
for p in expected: assert os.path.exists(p), p
assert os.path.isdir(RUNS_DIR) and len(os.listdir(RUNS_DIR)) == len(results_df)
print("OK. models:", sorted(os.listdir(MODELS_DIR)))
print("runs:", sorted(os.listdir(RUNS_DIR)))